# Project Lab 2: Hybrid Search, LoRA Fine-Tuning, Document Extraction

Notebook for **Chapter 16** of the *genai* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/genai/ch16.html

Tip: change a number, predict the output, then run the cell again.

## 16.1 Project: A Hybrid Semantic Search Engine With Recall@k

**Goal and why employers care.** Build a search engine over 12 short product and help documents that combines BM25 (keywords), vector similarity (meaning) and a reranker, then measure which combination finds the right document. Search is behind shop catalogs, help centres and RAG. Showing recall@k for four methods on queries that break each method is the exact evidence an interviewer wants.

In [ ]:
CONCEPTS = {"money": {"refund","reimbursed","money","back","payment","returns"},
            "bag": {"backpack","rucksack","bag","pack"}, "hike": {"hiking","hike","trek","trail","boots"},
            "light": {"lamp","lantern","light","tent","tents","camping","camp"}, ...}   # 9 concepts in all
CIDS = sorted(CONCEPTS)
def embed(text):
    v = np.zeros(len(CIDS))
    for w in tok(text):
        for i, c in enumerate(CIDS):
            if w in CONCEPTS[c]: v[i] += 1
    n = np.linalg.norm(v); return v / n if n else v
M = np.array([embed(t) for t in texts])
def vector_scores(q): return M @ embed(q)          # cosine, because vectors have length 1

In [ ]:
def minmax(x): return (x - x.min()) / (x.max() - x.min()) if x.max() > x.min() else np.zeros_like(x)
def fuse_weighted(q, alpha=0.5): return alpha * minmax(bm.scores(q)) + (1 - alpha) * minmax(vector_scores(q))
def fuse_rrf(q, k=60):
    out = np.zeros(len(ids))
    for sc in (bm.scores(q), vector_scores(q)):
        for r, j in enumerate(rank(sc)): out[j] += 1 / (k + r + 1)
    return out

## 16.2 Project: Fine-Tune a Small Model With LoRA on a Narrow Task

**Goal and why employers care.** Prepare data, plan memory, and design the evaluation for fine-tuning a small open model with LoRA to label support tickets (delivery, refund, product_defect, account). The training run itself needs a GPU, so it is "not run here". Most of the value, and most of the failures, are in the parts around training: bad data, leakage, and a weak evaluation. Employers want people who know when fine-tuning is the right tool and can budget its cost.

In [ ]:
def to_chat(r):
    return {"messages": [
        {"role": "system", "content": "Classify the ticket. Reply with exactly one of: " + ", ".join(LABELS)},
        {"role": "user", "content": r["text"]}, {"role": "assistant", "content": r["label"]}]}

In [ ]:
def dedupe(rows):
    seen, uniq = set(), []
    for r in rows:
        key = "".join(c for c in r["text"] if not c.isdigit())   # ignore the numbers
        if key not in seen: seen.add(key); uniq.append(r)
    return uniq, len(rows) - len(uniq)

In [ ]:
# NOT RUN HERE: needs a GPU, a model download and installed libraries.
from peft import LoraConfig, get_peft_model
cfg = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05,
                 target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM")
model = get_peft_model(base_model, cfg)      # base_model loaded from a small open model
model.print_trainable_parameters()           # should show well under 1 percent
# then: a trainer on train.jsonl, watch validation loss, save only the adapter

## 16.3 Project: A Document Extraction Pipeline (OCR Text In, Validated JSON Out)

**Goal and why employers care.** Take OCR text from shop invoices, get a model to return JSON with four fields (vendor, invoice number, date, total), validate it against a schema, repair known problems with code, and measure field-level accuracy. Companies have mountains of invoices, forms and letters. The skill that earns trust is validation and measurement, not a clever prompt.

In [ ]:
text = f"{shop}\nInvoice No: {inv}\nDate: {day:02d}/10/2026\nTotal: ${amt:,.2f}"
gold = {"vendor": shop, "invoice_no": inv, "date": f"2026-10-{day:02d}", "total": amt}
if i % 3 == 1: text = text.replace("Invoice No", "lnvoice No")   # OCR reads capital I as lowercase l
if i % 3 == 2: text = text.replace("Total", "Amount Due")        # different label wording
if i % 6 == 5: text = text.replace("0", "O")                     # OCR reads zero as letter O

In [ ]:
SCHEMA = {"vendor": str, "invoice_no": str, "date": str, "total": float}
def validate(obj):
    errs = []
    for k, t in SCHEMA.items():
        if k not in obj or obj[k] is None: errs.append(f"{k}: missing")
        elif not isinstance(obj[k], t): errs.append(f"{k}: expected {t.__name__}, got {type(obj[k]).__name__}")
    if not errs and not re.fullmatch(r"INV-\d{4}", obj["invoice_no"]): errs.append("invoice_no: bad format")
    if not errs and not re.fullmatch(r"\d{4}-\d\d-\d\d", obj["date"]): errs.append("date: bad format")
    return errs

In [ ]:
def repair(obj):
    if isinstance(obj.get("total"), str): obj["total"] = float(obj["total"].replace(",", ""))
    return obj

## 16.4 Checkpoints, Rubric and Five Debugging Problems for These Labs

The three labs above fail in their own ways. Use the checkpoints to know you are on track, the rubric to grade yourself like a reviewer, and the five problems to practise finding the cause of a bad number.

In [ ]:
import numpy as np
bm25 = np.array([8.2, 3.1, 0.4, 0.0])          # keyword scores, document 0 matches exact words
cosine = np.array([0.20, 0.35, 0.92, 0.88])    # meaning scores, document 2 is the paraphrase
raw = 0.5 * bm25 + 0.5 * cosine
mm = lambda x: (x - x.min()) / (x.max() - x.min())
scaled = 0.5 * mm(bm25) + 0.5 * mm(cosine)
print("raw sum order:   ", np.argsort(-raw).tolist())
print("scaled sum order:", np.argsort(-scaled).tolist())
print("raw scores:   ", np.round(raw, 3))
print("scaled scores:", np.round(scaled, 3))

raw sum order:    [0, 1, 2, 3]
scaled sum order: [2, 0, 3, 1]
raw scores:    [4.2   1.725 0.66  0.44 ]
scaled scores: [0.5   0.293 0.524 0.472]


In [ ]:
import re
q = "error E-4021 on card reader"
print("bad tokeniser: ", re.findall(r"[a-z0-9]+", q.lower()))
print("good tokeniser:", re.findall(r"[a-z0-9\-]+", q.lower()))

bad tokeniser:  ['error', 'e', '4021', 'on', 'card', 'reader']
good tokeniser: ['error', 'e-4021', 'on', 'card', 'reader']


In [ ]:
import numpy as np
y = np.array(["ok"] * 94 + ["defect"] * 6)
pred = np.array(["ok"] * 100)                       # a model that never predicts a defect
print("accuracy:", (y == pred).mean())
for label in ("ok", "defect"):
    m = y == label
    print(f"recall for {label:6s}:", (pred[m] == label).mean())

accuracy: 0.94
recall for ok    : 1.0
recall for defect: 0.0


In [ ]:
import re
train = ["Order 1001 has not arrived", "Refund for order 2002 please", "App crashes on login"]
test  = ["Order 3003 has not arrived", "Where is my parcel", "Refund for order 4004 please"]
mask = lambda s: re.sub(r"\d+", "#", s.lower())
exact = [t for t in test if t in train]
near = [t for t in test if mask(t) in {mask(x) for x in train}]
print("exact overlap:", exact)
print("near overlap (digits masked):", near)

exact overlap: []
near overlap (digits masked): ['Order 3003 has not arrived', 'Refund for order 4004 please']


In [ ]:
import re
text = "OMEGA TOOLS LTD   Total: $1O5.5O"
bad = text.replace("O", "0")
def fix_amount(s):
    m = re.search(r"Total: \$([0-9O.,]+)", s)
    return m.group(1).replace("O", "0") if m else None
print("blanket replace:", bad)
print("field-only fix: ", fix_amount(text), "| vendor kept:", text.split("  ")[0])

blanket replace: 0MEGA T00LS LTD   Total: $105.50
field-only fix:  105.50 | vendor kept: OMEGA TOOLS LTD


## Chapter recap

## Practice set

**Task 1 (easy).** A document is rank 1 in both BM25 and vector rankings. What is its RRF score with k = 60? **Answer:** 1/61 + 1/61 = 2/61 = 0.0328.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/genai/ch16.html